In [2]:
import pandas as pd
df = pd.read_csv('data/train.csv')
df.head()

,month_churn,nid,report_date,v_0,v_1,v_10,v_100,v_101,v_102,v_103,...,v_91,v_92,v_93,v_94,v_95,v_96,v_97,v_98,v_99,week_churn
0,0,10363,2024-09-08,45,48,344,7,5,5,0,...,0,0,45,48,0,3,7,7,6,0
1,0,3303,2024-10-27,35,42,47,45,43,45,0,...,0,1,46,275,1,1,47,47,45,1
2,0,9732,2023-10-29,35,39,19,19,6,6,0,...,0,0,45,76,1,0,19,19,19,1
3,1,2934,2024-12-08,11,15,101,91,31,48,0,...,0,1,122,460,0,0,93,93,84,1
4,0,3251,2024-03-10,3,3,181,33,23,23,0,...,0,0,29,73,1,2,33,33,32,0


In [3]:
import pandas as pd
import os
import numpy as np

# читаем исходный датасет
df = pd.read_csv("data/train.csv")

# считаем количество строк на каждый nid
nid_counts = df["nid"].value_counts().reset_index()
nid_counts.columns = ["nid", "count"]

# сортируем по количеству строк
nid_counts = nid_counts.sort_values("count").reset_index(drop=True)

# задаем желаемое количество частей
num_parts = 10  # можно поменять

# считаем общее количество строк и целевой размер части
total_rows = nid_counts["count"].sum()
target_rows = total_rows / num_parts

# разбиваем по логике "равных сумм строк", но сохраняя близкие count
parts = []
current_part = []
current_sum = 0

for _, row in nid_counts.iterrows():
    nid, count = row["nid"], row["count"]
    if current_sum + count > target_rows and len(parts) < num_parts - 1:
        parts.append(current_part)
        current_part = [nid]
        current_sum = count
    else:
        current_part.append(nid)
        current_sum += count

# добавляем последний кусок
if current_part:
    parts.append(current_part)

for i, part in enumerate(parts, 1):
    part_df = df[df["nid"].isin(part)]
    stats = part_df["nid"].value_counts()
    part_min = stats.min()
    part_max = stats.max()

    # считаем количество нулей
    week_zeros = (part_df["week_churn"] == 0).sum()
    month_zeros = (part_df["month_churn"] == 0).sum()

    print(f"\n=== part_{i} ===")
    print(f"Количество уникальных nid: {len(stats)}")
    print(f"Диапазон строк на nid: {part_min}–{part_max}")
    print(f"Всего строк: {len(part_df)}")
    print(f"Нулей в week_churn: {week_zeros} ({week_zeros / len(part_df) * 100:.2f}%)")
    print(f"Нулей в month_churn: {month_zeros} ({month_zeros / len(part_df) * 100:.2f}%)")



=== part_1 ===
Количество уникальных nid: 5356
Диапазон строк на nid: 1–12
Всего строк: 24810
Нулей в week_churn: 10429 (42.04%)
Нулей в month_churn: 12060 (48.61%)

=== part_2 ===
Количество уникальных nid: 1591
Диапазон строк на nid: 12–20
Всего строк: 24811
Нулей в week_churn: 15737 (63.43%)
Нулей в month_churn: 19994 (80.59%)

=== part_3 ===
Количество уникальных nid: 1081
Диапазон строк на nid: 20–26
Всего строк: 24800
Нулей в week_churn: 17731 (71.50%)
Нулей в month_churn: 22120 (89.19%)

=== part_4 ===
Количество уникальных nid: 851
Диапазон строк на nid: 26–32
Всего строк: 24804
Нулей в week_churn: 19019 (76.68%)
Нулей в month_churn: 22493 (90.68%)

=== part_5 ===
Количество уникальных nid: 684
Диапазон строк на nid: 32–41
Всего строк: 24792
Нулей в week_churn: 18729 (75.54%)
Нулей в month_churn: 22022 (88.83%)

=== part_6 ===
Количество уникальных nid: 544
Диапазон строк на nid: 41–50
Всего строк: 24792
Нулей в week_churn: 18906 (76.26%)
Нулей в month_churn: 22465 (90.61%)

=

In [5]:
len(parts)

10

## Ансамбль из бустов

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool
from sklearn.linear_model import LogisticRegression

In [ ]:
train = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")
test_features = [c for c in test_df.columns if c not in ['week_churn', 'month_churn', 'nid', 'report_date']]

In [ ]:
target = 'week_churn'
drop_cols = ['month_churn', 'nid', 'report_date']
features = [c for c in train.columns if c not in drop_cols + [target]]


catboost_params = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 3,
    'eval_metric': 'AUC',
    'loss_function': 'Logloss',
    'random_seed': 42,
    'early_stopping_rounds': 50,
    'verbose': 100
}

models = []
predictions_train_meta = []
predictions_val_meta = []

for i, df in enumerate(list_of_dataframes):
    print(f"\n--- Обучение модели {i+1} ---")
    X = df[features]
    y = df[target]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    train_pool = Pool(X_train, y_train)
    val_pool = Pool(X_val, y_val)

    model = CatBoostClassifier(**catboost_params)
    model.fit(train_pool, eval_set=val_pool)

    models.append(model)

    y_pred_proba_val = model.predict_proba(X_val)[:, 1]
    predictions_val_meta.append(y_pred_proba_val)

    predictions_train_meta.append(y_pred_proba_val)

    roc_auc = roc_auc_score(y_val, y_pred_proba_val)
    print(f"ROC-AUC модели {i+1} на валидации: {roc_auc:.4f}")


y_val_combined = []
for df in list_of_dataframes:
    X_temp = df[features]
    y_temp = df[target]
    _, X_v_temp, _, y_v_temp = train_test_split(
        X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
    )
    y_val_combined.extend(y_v_temp.values)

X_meta = np.column_stack(predictions_train_meta)
y_meta = np.array(y_val_combined)

print(f"\nРазмерность X_meta для мета-модели: {X_meta.shape}")
print(f"Размерность y_meta для мета-модели: {y_meta.shape}")

meta_model = LogisticRegression(random_state=42)
meta_model.fit(X_meta, y_meta)

print("\nМета-модель обучена.")

In [ ]:
test_predictions = []
for model in models:
    pred_proba = model.predict_proba(X_test)[:, 1]
    test_predictions.append(pred_proba)

# Формируем признаки для мета-модели из тестовых предсказаний
X_test_meta = np.column_stack(test_predictions)

# Получаем итоговое предсказание от мета-модели
final_predictions_proba = meta_model.predict_proba(X_test_meta)[:, 1] # Вероятности класса 1
final_predictions = meta_model.predict(X_test_meta) # Бинарные предсказания


from sklearn.model_selection import KFold
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
oof_predictions = np.zeros((len(X_train_full), len(models))) # Предполагаем X_train_full, y_train_full
for fold_n, (train_idx, val_idx) in enumerate(kf.split(X_train_full)):
    X_tr, X_val = X_train_full.iloc[train_idx], X_train_full.iloc[val_idx]
    y_tr, y_val = y_train_full.iloc[train_idx], y_train_full.iloc[val_idx]
    for model_idx, base_model in enumerate(models):
        # Обучаем копию базовой модели на fold
        temp_model = CatBoostClassifier(**catboost_params)
        temp_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        # Предсказываем на val_idx
        oof_predictions[val_idx, model_idx] = temp_model.predict_proba(X_val)[:, 1]
# Затем обучаем meta_model на oof_predictions и y_train_full
meta_model.fit(oof_predictions, y_train_full)
